In [9]:
from typing import Self
from typing import Iterator

In [10]:
class Node:
    def __init__(self, data):
        self.data = data
        self.right_child = None
        self.left_child = None

    def __repr__(self) -> str:
        return f"{'+' if self.left_child is not None else '-'}:{self.data}:{'+' if self.right_child is not None else '-'}"

    def __str__(self) -> str:
        return repr(self)

    def __len__(self) -> None:
        return sum([self.left_child is not None, self.right_child is not None])

    def __lt__(self, other: Self) -> bool:
        return self.data < other.data

    def __le__(self, other: Self) -> bool:
        return self.data <= other.data

    def __eq__(self, other: Self) -> bool:
        return self.data == other.data

In [11]:
class InOrderTreeIterator:
    def __init__(self, root: Node) -> None:
        self.stack = []
        self._push_left(node=root)

    def _push_left(self, node: Node):
        while node is not None:
            self.stack.append(node)
            node = node.left_child

    def has_next(self) -> bool:
        return bool(self.stack)

    def __iter__(self):
        return self

    def __next__(self):
        if not self.has_next():
            raise StopIteration

        node = self.stack.pop()

        if node.right_child is not None:
            self._push_left(node.right_child)
        return node

In [38]:
class TreeIterator:
    def __init__(self, root: Node) -> None:
        self.stack = []
        self.fill(node=root)

    def fill(self, node: Node):
        def go(node: None) -> None:
            self.stack.append(node)
            if node.left_child is not None:
                go(node=node.left_child)
            if node.right_child is not None:
                go(node=node.right_child)

        go(node=node)

    def __iter__(self):
        return self

    def __next__(self):
        if len(self.stack) == 0:
            raise StopIteration

        node = self.stack.pop()

        return node

In [ ]:
class Tree:
    def __init__(self):
        self.root_node = None

    def is_valid(self) -> bool:
        stack = [(self.root_node, None)]
        while len(stack) > 0:
            node, parent = stack.pop()

            if len(node) == 0:
                continue

            if node.left_child is not None:
                if node.left_child < node:
                    if parent is not None:
                        if parent < node:
                            if node.left_child < parent:
                                return False

                    stack.append((node.left_child, node))

                else:
                    return False

            if node.right_child is not None:
                if node <= node.right_child:
                    if parent is not None:
                        if node < parent:
                            if parent < node:
                                return False
                    stack.append((node.right_child, node))
                else:
                    return False

        return True

    def iter(self) -> Iterator[Node]:
        return TreeIterator(root=self.root_node)

    def insert(self, data):
        node = Node(data)
        if self.root_node is None:
            self.root_node = node
            return self.root_node
        else:
            current = self.root_node
            parent = None
            while True:
                parent = current
                if node.data < parent.data:
                    current = current.left_child
                    if current is None:
                        parent.left_child = node
                        return self.root_node
                else:
                    current = current.right_child
                    if current is None:
                        parent.right_child = node
                        return self.root_node

    def inorder(self, node: Node | None = None):
        if node is None:
            assert self.root_node is not None
            node = self.root_node
        if node.left_child is not None:
            self.inorder(node=node.left_child)
        print(node.data)
        if node.right_child is not None:
            self.inorder(node=node.right_child)

    def get_node_with_parent(self, data):
        parent = None
        current = self.root_node
        if current is None:
            return (parent, None)
        while True:
            if current.data == data:
                return (parent, current)
            elif current.data > data:
                parent = current
                current = current.left_child
            else:
                parent = current
                current = current.right_child
        return (parent, current)

    def remove(self, data):
        parent, node = self.get_node_with_parent(data)

        if parent is None and node is None:
            return False

        # Get children count
        children_count = 0

        if node.left_child and node.right_child:
            children_count = 2
        elif (node.left_child is None) and (node.right_child is None):
            children_count = 0
        else:
            children_count = 1

        if children_count == 0:
            if parent:
                if parent.right_child is node:
                    parent.right_child = None
                else:
                    parent.left_child = None
            else:
                self.root_node = None
        elif children_count == 1:
            next_node = None
            if node.left_child:
                next_node = node.left_child
            else:
                next_node = node.right_child

            if parent:
                if parent.left_child is node:
                    parent.left_child = next_node
                else:
                    parent.right_child = next_node
            else:
                self.root_node = next_node
        else:
            parent_of_leftmost_node = node
            leftmost_node = node.right_child
            while leftmost_node.left_child:
                parent_of_leftmost_node = leftmost_node
                leftmost_node = leftmost_node.left_child
            node.data = leftmost_node.data

            if parent_of_leftmost_node.left_child == leftmost_node:
                parent_of_leftmost_node.left_child = leftmost_node.right_child
            else:
                parent_of_leftmost_node.right_child = leftmost_node.right_child

    def search(self, data):
        current = self.root_node
        while True:
            if current is None:
                print("Item not found")
                return None
            elif current.data is data:
                print("Item found", data)
                return data
            elif current.data > data:
                current = current.left_child
            else:
                current = current.right_child

    def find_min(self):
        current = self.root_node
        while current.left_child:
            current = current.left_child
        return current.data

    def find_max(self):
        current = self.root_node
        while current.right_child:
            current = current.right_child
        return current.data

In [59]:
tree = Tree()
r = tree.insert(5)
r = tree.insert(3)
r = tree.insert(1)
r = tree.insert(4)
r = tree.insert(7)
r = tree.insert(9)
r = tree.insert(6)
r = tree.insert(12)

tree.inorder()

1
3
4
5
6
7
9
12


In [41]:
tree.root_node

+:5:+

In [42]:
for node in tree.iter():
    print(node)

-:12:-
-:9:+
-:6:-
+:7:+
-:4:-
-:1:-
+:3:+
+:5:+


In [43]:
tree.is_valid()

True

In [ ]:
# tree = Tree()
# r = tree.insert(5)
# r = tree.insert(2)
# r = tree.insert(7)
# r = tree.insert(7)
# r = tree.insert(7)
# r = tree.insert(9)
# r = tree.insert(1)

# tree.root_node.right_child = Node(3)

# tree.is_valid()

False

In [47]:
tree.search(7), tree.search(5)

Item found 7
Item found 5


(7, 5)

In [20]:
tree.search(9)

Item found 9


9

In [49]:
tree.remove(9)
tree.search(9)

Item not found


In [50]:
tree = Tree()
tree.insert(5)
tree.insert(2)
tree.insert(7)
tree.insert(9)
tree.insert(1)

print(tree.find_min())
print(tree.find_max())

2
7


In [79]:
import dis

x = 2

dis.dis('eval("x + 3")')

  0           RESUME                   0

  1           LOAD_NAME                0 (eval)
              PUSH_NULL
              LOAD_CONST               0 ('x + 3')
              CALL                     1
              RETURN_VALUE
